### 0. Set-up

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "rag"))
from rewrite import rewrite
from search import search, expand_terms, count_drugs
from answer import (ANSWER_PROMPT, MODEL, build_context, check, client, _clean, finalize, check_sources)

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

### 1. Question

In [2]:
#question = "Which EGFR inhibitors are being tested in lung cancer?"
#question = "Which lung cancer trials is AstraZeneca currently running?"
#question = "Are there recruiting trials for Parkinson's disease?"
question = "Keytruda trials in melanoma"

terms = rewrite(question)
print("=== FROM THE LLM ===")
for key, value in terms.items():
    print(f"{key}: {value}")

terms["interventions"] = expand_terms(terms["interventions"])
print("\n=== INTERVENTIONS AFTER ADDING NAMES FROM THE DATA ===")
for group, names in terms["interventions"].items():
    print(f"{group}: {names}")

=== FROM THE LLM ===
focus: Conditions: melanoma. Interventions: pembrolizumab (Keytruda).
conditions: ['melanoma', 'malignant melanoma']
interventions: {'pembrolizumab': ['pembrolizumab', 'Keytruda', 'MK-3475']}
status: None
sponsors: []

=== INTERVENTIONS AFTER ADDING NAMES FROM THE DATA ===
pembrolizumab: ['pembrolizumab', 'Keytruda', 'MK-3475', 'bcd-201', 'gme 751', 'gme751', 'keytruda, mk-3475', 'lambrolizumab', 'mk 3475', 'mk-3475,', 'mk-3475, keytruda', 'mk-3475-d79', 'mk3475', 'ql2107', 'rph 075', 'rph-075', 'rph075', 'sb 27', 'sb-27', 'sb27', 'sch 900475', 'sch-900475', 'sch900475']


### 2. Search

In [3]:
selection = dict(conditions=terms["conditions"], interventions=terms["interventions"],
                 status_group=terms["status"], sponsors=terms["sponsors"])

trials = search(focus=terms["focus"], k=15, **selection)
total = int(trials["total_matches"].iloc[0]) if len(trials) else 0

print("Selected trials:", total, "| shown:", len(trials), "\n")
print(trials[["trial_id", "title", "status", "sponsor", "similarity"]]
      .round({"similarity": 2}).to_string(index=False))

W1116 17:54:39.398000 1060 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Selected trials: 59 | shown: 15 

   trial_id                                                                                                                                                                                                   title                status                       sponsor  similarity
NCT04305054  Substudy 02B: Safety and Efficacy of Pembrolizumab in Combination With Investigational Agents or Pembrolizumab Alone in Participants With First Line (1L) Advanced Melanoma (MK-3475-02B/KEYMAKER-U02)             COMPLETED       Merck Sharp & Dohme LLC        0.71
NCT03820986          Safety and Efficacy Study of Pembrolizumab (MK-3475) Combined With Lenvatinib (MK-7902/E7080) as First-line Intervention in Adults With Advance Melanoma (MK-7902-003/E7080-G000-312/LEAP-003)             COMPLETED       Merck Sharp & Dohme LLC        0.69
NCT06697301                                                                 Safety and Efficacy of EIK1001 in Combo With Pembro Versus Pla

### 3. Counts

In [4]:
counts = count_drugs(**selection)
print(counts.to_string(index=False))

                    drug                                               other_names  trials
           pembrolizumab            ql2107, mk-3475, keytruda, mk-3475-d79, mk3475      55
              lenvatinib                         dtc cohort, e 7080, e-7080, e7080       7
              ipilimumab            anti-ctla-4, bms 734016, bms-734016, bms734016       4
               nivolumab  bms-936558-01, opdivo,, anti-programmed-death-1, bcd-263       4
            vibostolimab                                                   mk-7684       4
                    atra                                                       NaN       3
     intismeran autogene                                           mrna-4157, v940       3
             quavonlimab                                                   mk-1308       3
               8 agonist                                                       NaN       2
            atezolizumab              mpdl 3280a, mpdl 328oa, mpdl-3280a, mpdl3280       2

### 4. Message send to LLM

In [5]:
context = build_context(terms, trials, total, counts)
user_message = f"Question: {question}\n\n{context}"

print("=== SYSTEM MESSAGE (the instructions) ===\n")
print(ANSWER_PROMPT)
print("\n=== USER MESSAGE (the question and the sources) ===\n")
print(user_message)

=== SYSTEM MESSAGE (the instructions) ===

You answer questions about clinical trials with at least one site in Australia,
using data from ClinicalTrials.gov.

## What you are given
1. Selection: which trials were selected (status and sponsor, if the question named them).
2. Counts: every drug that appears in ALL selected trials, with the number of trials.
   This includes drugs that do not fit the question (for example chemotherapy given
   alongside, or another drug tested in the same patients).
3. Trials: the most relevant selected trials, with details. This is a sample, not all trials.

## How to use it
- Base every statement on the information given. Use your own knowledge only to recognise
  that two names refer to the same drug.
- Decide which drugs in the counts fit the question; mention only those.
- For numbers, use the counts and the total, never the sample.
- For examples, use the trials: at most two per drug.
- Leave out trials that do not fit the question.

## How to answ

### 5. What LLM sends back (the draft)

In [6]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "system", "content": ANSWER_PROMPT},
              {"role": "user", "content": user_message}],
    temperature=0,
    reasoning_effort="medium",
    max_tokens=5000,
)
draft = _clean(response.choices[0].message.content)
print(draft)

RateLimitError: Error code: 429 - Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m76z3ffrej5t817jw5psbyxp` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198733, Requested 4709. Please try again in 24m46.944s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing

### 7. The check + final answer

In [ ]:
sources = check_sources(terms, trials, total, counts, draft)
checked = check(question, sources, draft)
final, changes = finalize(draft, checked, trials)

print("=== CHANGES BY THE CHECK ===")
for c in changes or ["(none)"]:
    print("-", c)
print("\n=== FINAL ANSWER ===\n")
print(final)